# **OOP Project Completion and Presentation**

**DS212 — Object-Oriented Programming**

**Week 17:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Refine, document, package, and present the completed object-oriented application.

In the final week, the project moves from *"it works on my laptop"* to a **finished product**: integrated, tested, documented, packaged so others can reproduce it, and presented clearly.

```{mermaid}
flowchart LR
    I["Integrate & test"] --> D["Document"]
    D --> P["Package & reproduce"]
    P --> R["Rehearse"]
    R --> S["Present & demonstrate"]
```

## **Integration, Testing, and Debugging**

### **Unit tests vs. integration tests**

| Test type            | Checks                                             | Example                                                  |
| -------------------- | -------------------------------------------------- | -------------------------------------------------------- |
| **Unit test**        | one class or method in isolation                   | `Order.total()` adds line subtotals                      |
| **Integration test** | several classes working together                   | place orders → advance statuses → daily report is correct |
| **End-to-end check** | the whole application as a user runs it            | the demo notebook runs from top to bottom                |

Integration problems often appear only when real pieces are connected: mismatched method names, data in an unexpected format, or an object in the wrong state. An **integration test** walks through a complete scenario:

In [ ]:
import sys
import unittest
from collections import defaultdict


class Item:
    def __init__(self, code, price):
        self.code, self.price = code, price


class Order:
    def __init__(self, lines):
        self.lines = lines                         # list of (Item, qty)
        self.claimed = False

    def total(self):
        return sum(item.price * qty for item, qty in self.lines)


class SalesReport:
    def __init__(self, orders):
        self.quantities = defaultdict(int)
        self.revenue = defaultdict(float)
        for order in orders:
            if not order.claimed:                  # only completed sales count
                continue
            for item, qty in order.lines:
                self.quantities[item.code] += qty
                self.revenue[item.code] += item.price * qty

    def total_revenue(self):
        return sum(self.revenue.values())


class TestDailyScenario(unittest.TestCase):
    def test_report_counts_only_claimed_orders(self):
        rice, adobo = Item("RICE", 15), Item("ADOBO", 80)
        o1 = Order([(rice, 2), (adobo, 1)])
        o2 = Order([(adobo, 2)])
        o3 = Order([(rice, 1)])
        o1.claimed = o2.claimed = True             # o3 was never claimed

        report = SalesReport([o1, o2, o3])

        self.assertEqual(report.quantities["ADOBO"], 3)
        self.assertEqual(report.quantities["RICE"], 2)
        self.assertEqual(report.total_revenue(), o1.total() + o2.total())


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestDailyScenario)
unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(suite);

### **Final quality pass**

* Run the **full test suite** from a fresh clone of the repository.
* Run the **demo notebook** with *Kernel → Restart & Run All*.
* Try **invalid inputs** on purpose — the application should respond with clear messages, never a raw traceback.
* Remove dead code, debugging `print()` calls, and unused files.

## **Documentation, Packaging, and Reproducibility**

### **Docstrings**

Every public module, class, and method should have a **docstring** (PEP 257). A consistent style such as the Google style makes them readable in editors and documentation tools:

In [ ]:
class Order:
    '''A customer's order made of menu items and quantities.

    Attributes:
        order_id: Sequential number assigned by the canteen.
        customer: Name of the customer who placed the order.
    '''

    def __init__(self, order_id, customer):
        self.order_id = order_id
        self.customer = customer
        self._lines = []

    def add_line(self, item, quantity):
        '''Add an item to the order.

        Args:
            item: The MenuItem being ordered.
            quantity: Number of units; must be at least 1.

        Raises:
            ValueError: If quantity is less than 1.
        '''
        if quantity < 1:
            raise ValueError("Quantity must be at least 1.")
        self._lines.append((item, quantity))


help(Order.add_line)

### **Project documentation**

| Document                 | Audience          | Contents                                                       |
| ------------------------ | ----------------- | -------------------------------------------------------------- |
| `README.md`              | everyone          | purpose, installation, quick-start example, how to run tests   |
| docstrings               | developers        | what each class and method does, parameters, exceptions        |
| design document          | developers, panel | final class and sequence diagrams; changes from the proposal and **why** |
| demo notebook            | users, panel      | a guided, runnable walkthrough of the main scenarios           |
| `CHANGELOG.md`           | users             | what changed in each version                                   |

### **Reproducibility checklist**

Someone else — your instructor, a panel member, or you in six months — must be able to run the project **from scratch**:

* [ ] `git clone` → `python -m venv .venv` → `pip install -e .` works on a clean machine;
* [ ] dependencies are declared in `pyproject.toml` (and pinned in `requirements.txt` for the demo);
* [ ] sample data needed by the demo is included (with **no** real personal data) or generated by a script;
* [ ] random processes use a fixed **seed**;
* [ ] file paths are relative to the project (no `C:\Users\...` paths);
* [ ] the release is tagged in Git, and the built wheel installs correctly.

## **Presentation and Demonstration**

### **Structure**

A clear project presentation tells a story, not a list of files:

| Part              | Content                                                          | Time   |
| ----------------- | ---------------------------------------------------------------- | ------ |
| 1. Problem        | who has the problem, why it matters                              | 1–2 min |
| 2. Solution       | what the application does (requirements in one slide)            | 1–2 min |
| 3. Design         | class diagram; how the four core concepts and relationships are used | 3 min |
| 4. Live demo      | the main scenario, plus one error being handled gracefully       | 3–4 min |
| 5. Quality        | tests, packaging, Git workflow                                   | 1–2 min |
| 6. Reflection     | what was hard, what you would change, what you learned           | 1 min  |

### **Demo tips**

* **Script** the demo and rehearse it — know exactly which cells or commands you will run.
* Prepare **sample data** in advance; do not type long inputs live.
* Show **one failure case** on purpose (invalid input) — it demonstrates robustness.
* Have a **backup**: screenshots or a recording in case the live demo fails.
* Increase the font size; close unrelated windows and notifications.

### **Explaining your design**

Panels often ask design questions. Be ready to answer, with reference to your diagram:

* *Why is this an inheritance relationship and not composition?*
* *Which class is responsible for validating this data, and why?*
* *What happens if a user enters an invalid value here?*
* *How would you add a new feature, such as another report format? Which classes would change?*
* *How do you know it works? Show a test.*

## **Final Submission Checklist**

* [ ] source code in a Git repository with a meaningful commit history and a release tag;
* [ ] `pyproject.toml`, `README.md`, `LICENSE`, and `CHANGELOG.md`;
* [ ] passing unit and integration tests;
* [ ] a demo notebook that runs from top to bottom;
* [ ] final UML diagrams and a short design document;
* [ ] presentation slides;
* [ ] references and an AI-use acknowledgement, if applicable.

## **Summary**

| Activity            | Goal                                                              |
| ------------------- | ----------------------------------------------------------------- |
| integration testing | verify that classes work together across complete scenarios       |
| documentation       | make the project understandable to users and developers           |
| packaging           | make the project installable and reproducible                     |
| presentation        | communicate the problem, design, quality, and lessons learned     |

## **References**

Doumont, J.-L. (2009). *Trees, maps, and theorems: Effective communication for rational minds*. Principiae.

Goodger, D., & van Rossum, G. (2001). *PEP 257 — Docstring conventions*. Python Software Foundation. https://peps.python.org/pep-0257/

Make a README. (2026). *Make a README*. https://www.makeareadme.com/

Python Packaging Authority. (2026). *Python packaging user guide*. https://packaging.python.org/

Python Software Foundation. (2026). *unittest — Unit testing framework*. https://docs.python.org/3/library/unittest.html

Thomas, D., & Hunt, A. (2019). *The pragmatic programmer: Your journey to mastery* (20th anniversary ed.). Addison-Wesley.

Wilson, G., Aruliah, D. A., Brown, C. T., Chue Hong, N. P., Davis, M., Guy, R. T., Haddock, S. H. D., Huff, K. D., Mitchell, I. M., Plumbley, M. D., Waugh, B., White, E. P., & Wilson, P. (2014). Best practices for scientific computing. *PLOS Biology, 12*(1), e1001745. https://doi.org/10.1371/journal.pbio.1001745

Wilson, G., Bryan, J., Cranston, K., Kitzes, J., Nederbragt, L., & Teal, T. K. (2017). Good enough practices in scientific computing. *PLOS Computational Biology, 13*(6), e1005510. https://doi.org/10.1371/journal.pcbi.1005510